# Module 8: Capstone — Decision-Memo System — Deploy to Production

**Capstone — all four patterns**: Orchestrator (P5 Agent-as-Tool) delegates to Researcher (P1), parallel analyzers (P2 via asyncio.gather), and Critic-Refiner (P3). All patterns run inside one Runtime container.

![Capstone — Decision-Memo System Runtime on Amazon Bedrock AgentCore](./architecture.png)

This notebook walks you through the complete deployment cycle:
1. Install the AgentCore CLI
2. Configure the project
3. Review `main.py` — the Runtime entry point
4. Deploy to Amazon Bedrock AgentCore Runtime
5. Test the deployed agent
6. Clean up

**Prerequisites:** AWS credentials with Bedrock and AgentCore permissions. The IAM policy is in `../../../static/iam_policy.json`.

---

## Step 1 — Install the AgentCore CLI

In [ ]:
!uv pip install -q bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents nest-asyncio

---

## Step 2 — Configure the Project

Run these commands in a terminal from the `production/` folder.

> **Run in terminal, not in the notebook** — `agentcore` is a CLI tool, not a Python API.

In [ ]:
# Run this block as reference — execute the commands in your terminal

print("""
Navigate to the production folder:

  cd samples/08-capstone/production

Create the AgentCore project:

  agentcore create
  # Enter a name when prompted, e.g.: 08-capstone-runtime

Add the agent:

  agentcore add
  # Choose: agent
  # Choose: Bring my own code
  # Entrypoint file: main.py
  # Deploy mode: Direct Code Deploy

Copy the runtime files into the agent folder:

  cp main.py mock_tools.py requirements.txt app/<AgentName>/

Set up dependencies:

  cd app/<AgentName>
  uv init --bare --python 3.13
  uv add strands-agents bedrock-agentcore aws-opentelemetry-distro nest-asyncio
  cd ../..
""")

---

## Step 3 — Review `main.py`

This is the code that runs inside the Runtime container.

In [ ]:
print(open("main.py").read())

---

## Step 4 — Deploy

Run `agentcore deploy` from the `production/` folder. First deploy takes 3–5 minutes.

In [ ]:
print("""
Run in terminal from the production/ folder:

  agentcore deploy

The CLI will:
  1. Package your code and dependencies
  2. Upload to an Amazon S3 staging bucket
  3. Build a container image in Amazon ECR
  4. Create the Amazon Bedrock AgentCore Runtime
  5. Set up the Runtime endpoint

Watch for: 'Deployment successful' and the Runtime ARN in the output.
""")

---

## Step 5 — Test the Deployed Agent

In [ ]:
print("""
Test with the agentcore CLI (run in terminal):

  agentcore invoke "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch. Target: +15% CLV in 6 months. Budget: $2M."

Expected: a leadership memo with Recommendation, Options A/B/C table, Risks, Metrics, Decision Required.
""")

---

## Step 6 — Local Testing

Test the agent locally before or after deployment — no AWS resources needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

# Local test — runs main.py's logic without the AgentCore Runtime wrapper
from main import invoke

result = invoke(
    {"prompt": "NovaCart Premium Tier: Options A ($19.99), B ($14.99), C ($12.99). Target +15% CLV."},
    context={}
)
print(result["response"])

---

## Step 7 — Observability

After deployment, traces appear in CloudWatch automatically.

In [ ]:
print("""
View traces in CloudWatch:

  AWS Console → CloudWatch → X-Ray → Traces
  or
  AWS Console → Amazon Bedrock → AgentCore → Observability

What you will see:
  - One root span per invoke call
  - Child spans per Agent() call inside the pipeline
  - Tool call spans nested under each agent
  - Duration breakdown per stage

No extra configuration needed — aws-opentelemetry-distro is in requirements.txt
and agentcore deploy installs it automatically.
""")

---

## Step 8 — Cleanup

Two steps required — both must run.

In [ ]:
print("""
Run in terminal from the production/ folder:

  # Step 1: Reset local config (does NOT touch AWS)
  agentcore remove all -y

  # Step 2: Delete the Runtime from AWS
  agentcore deploy

Verify cleanup:

  aws bedrock-agentcore list-agent-runtimes --region us-east-1
""")
